# Actor-Critic

**Actor-Critic** adalah arsitektur hibrida yang menggabungkan keunggulan metode **Value-Based** seperti Q-Learning/DQN dan **Policy-Based** seperti REINFORCE/Policy Gradient.

Pendekatan murni Policy Gradient seperti REINFORCE mengalami *high variance* karena harus menunggu 1 episode selesai untuk menghitung return kumulatif (`R_t`).

Actor-Critic mengatasi masalah tersebut dengan menggunakan **Critic** untuk memberikan feedback di setiap langkah melalui **Temporal Difference (TD) error**.

## Dua Komponen Utama

### 1. Actor (`πθ(a | s)`)

* **Tugas:** Memilih aksi berdasarkan state saat ini.
* **Output:** Distribusi probabilitas aksi (stokastik) atau aksi langsung (deterministik).
* **Pembaharuan:** Menggunakan *gradient ascent* berdasarkan evaluasi dari Critic.

### 2. Critic (`Vφ(s)` atau `Qφ(s, a)`)

* **Tugas:** Menilai seberapa baik state atau aksi yang dipilih oleh Actor.
* **Output:** Nilai *state value* `V(s)` atau *action value* `Q(s, a)`.
* **Pembaharuan:** Menggunakan Mean Squared Error (MSE) berdasarkan TD error.

$$
\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
$$

## Advantage Function

Alih-alih menggunakan *raw reward* atau Q-value secara langsung, varian modern Actor-Critic menggunakan **Advantage Function**.

$$
A(s_t, a_t) = Q(s_t, a_t) - V(s_t)
$$

Dalam praktiknya, Advantage dapat diperkirakan menggunakan TD error:

$$
A(s_t, a_t) \approx r_t + \gamma V(s_{t+1}) - V(s_t)
$$

* Jika `A(s, a) > 0`: Aksi tersebut **lebih baik** dari rata-rata → Actor menaikkan probabilitas aksi tersebut.
* Jika `A(s, a) < 0`: Aksi tersebut **lebih buruk** dari rata-rata → Actor menurunkan probabilitas aksi tersebut.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import gymnasium as env_lib

# 1. Jaringan Terpadu Actor-Critic (Shared Backbone)
class ActorCriticNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorCriticNetwork, self).__init__()
        # Shared feature extractor
        self.fc = nn.Linear(state_dim, 128)

        # Actor Head (Output: Probabilitas Aksi)
        self.actor_head = nn.Linear(128, action_dim)

        # Critic Head (Output: State Value V(s))
        self.critic_head = nn.Linear(128, 1)

    def forward(self, x):
        x = F.relu(self.fc(x))

        # Probabilitas aksi via Softmax
        action_probs = F.softmax(self.actor_head(x), dim=-1)

        # Nilai V(s)
        state_value = self.critic_head(x)
        return action_probs, state_value

# 2. Setup Environment & Model
env = env_lib.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

model = ActorCriticNetwork(state_dim, action_dim)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
gamma = 0.99
num_episodes = 500

# 3. Training Loop A2C
for episode in range(num_episodes):
    state, _ = env.reset()
    done = False
    episode_reward = 0

    while not done:
        state_tensor = torch.FloatTensor(state)
        probs, state_value = model(state_tensor)

        # Sampling aksi dari Actor
        dist = Categorical(probs)
        action = dist.sample()

        # Eksekusi aksi di Environment
        next_state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        episode_reward += reward

        # Dapatkan V(s') dari Critic untuk next_state
        next_state_tensor = torch.FloatTensor(next_state)
        _, next_state_value = model(next_state_tensor)

        # Hitung TD Target & Advantage
        # Jika episode selesai (done), V(s') = 0
        target_value = reward + (0 if done else gamma * next_state_value.item())
        advantage = target_value - state_value

        # --- Loss Calculation ---
        # 1. Actor Loss: -log_prob * Advantage
        actor_loss = -dist.log_prob(action) * advantage.detach()

        # 2. Critic Loss: MSE Loss antara V(s) dan TD Target
        critic_loss = F.mse_loss(state_value, torch.tensor([target_value]))

        # Total Loss
        total_loss = actor_loss + critic_loss

        # Backpropagation
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        state = next_state

    if (episode + 1) % 50 == 0:
        print(f"Episode {episode + 1}/{num_episodes} - Total Reward: {episode_reward}")

env.close()

Episode 50/500 - Total Reward: 11.0
Episode 100/500 - Total Reward: 62.0
Episode 150/500 - Total Reward: 35.0
Episode 200/500 - Total Reward: 32.0
Episode 250/500 - Total Reward: 301.0
Episode 300/500 - Total Reward: 14.0
Episode 350/500 - Total Reward: 115.0
Episode 400/500 - Total Reward: 101.0
Episode 450/500 - Total Reward: 280.0
Episode 500/500 - Total Reward: 20.0


## Spektrum Algoritma Berbasis Actor-Critic

- **A2C (Advantage Actor-Critic):** Varian *synchronous* yang efisien dan stabil.
- **A3C (Asynchronous Advantage Actor-Critic):** Menggunakan banyak agen paralel secara asinkron untuk mempercepat pengumpulan data.
- **PPO (Proximal Policy Optimization):** Menggunakan arsitektur Actor-Critic dengan *clipped objective* untuk mencegah perubahan policy yang terlalu drastis.
- **SAC (Soft Actor-Critic):** Ditujukan untuk ruang aksi kontinyu dengan menambahkan *entropy maximization* agar agen tetap aktif melakukan eksplorasi.